# TradeFlow AI — nb3_olm_finetune (v6 — Production Stable)

**Architecture**: `allenai/olmOCR-2-7B-1025` → **Qwen2.5-VL** (not Qwen2-VL!)
**Quantization**: NF4 4-bit via BitsAndBytesConfig
**LoRA**: rank=32, alpha=64 via PEFT (PRD §10.4)
**Trainer**: HuggingFace standard Trainer

> **Critical**: olmOCR-2-7B-1025 uses **Qwen2.5-VL** architecture.
> Must use `Qwen2_5_VLForConditionalGeneration`, NOT `Qwen2VLForConditionalGeneration`.
> The vision encoder MLP structure is different (SwiGLU vs simple FC).


In [ ]:
!pip install -q -U transformers peft datasets accelerate bitsandbytes trl qwen-vl-utils
!pip install -q pdf2image pillow python-dateutil
!apt-get update -qq && apt-get install -qq poppler-utils


In [ ]:
import json, os, gc
import torch
import numpy as np
from pathlib import Path
from PIL import Image
from pdf2image import convert_from_path
from datasets import Dataset
from kaggle_secrets import UserSecretsClient
from transformers import (
    AutoProcessor,
    Qwen2_5_VLForConditionalGeneration,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# === CONFIG (sesuai PRD v5.2) ===
MODEL_ID  = 'allenai/olmOCR-2-7B-1025'     # PRD §4 Decision 2 — Qwen2.5-VL architecture
LORA_R    = 32                               # PRD §10.4
LORA_ALPHA = 64
OUT_DIR   = Path('./olmocr-tradeflow-lora')
OUT_DIR.mkdir(parents=True, exist_ok=True)

NB0_INPUT     = Path('/kaggle/input/nb0-real-doc-augmentation')
NB1_INPUT     = Path('/kaggle/input/nb1-synthetic-generator')
MANIFEST_PATH = NB0_INPUT / 'dataset' / 'augmented_manifest.json'
SYNTHETIC_DIR = NB1_INPUT / 'dataset' / 'synthetic'
GT_PATH       = Path('/kaggle/input/tradeflow-real-docs/TradeFlow_GroundTruth_v5.2.json')

try:
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret('HF_TOKEN')
except:
    hf_token = None
    print('Warning: HF_TOKEN not found.')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {DEVICE}')
if DEVICE == 'cuda':
    for i in range(torch.cuda.device_count()):
        print(f'GPU {i}  : {torch.cuda.get_device_name(i)}')
        mem = torch.cuda.get_device_properties(i).total_memory
        print(f'VRAM {i} : {mem / 1e9:.1f} GB')


## 1. Load Model (Qwen2.5-VL + 4-bit NF4 + LoRA r=32)

In [ ]:
print(f'Loading {MODEL_ID} (Qwen2.5-VL architecture)...')

# 4-bit NF4 quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

processor = AutoProcessor.from_pretrained(MODEL_ID, token=hf_token)

# Use Qwen2_5_VLForConditionalGeneration to auto-detect the correct class (Qwen2_5_VLForConditionalGeneration)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    token=hf_token,
)
print(f'Model class: {type(model).__name__}')  # Should show Qwen2_5_VLForConditionalGeneration

# Prepare for k-bit training
model = prepare_model_for_kbit_training(model)

# LoRA config — PRD §10.4: rank=32
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=['q_proj', 'v_proj', 'k_proj', 'o_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Verify VRAM
if DEVICE == 'cuda':
    allocated = torch.cuda.memory_allocated() / 1e9
    reserved  = torch.cuda.memory_reserved() / 1e9
    print(f'VRAM allocated: {allocated:.2f} GB | reserved: {reserved:.2f} GB')


## 2. Dataset Preparation (Multimodal)

In [ ]:
EXTRACTION_PROMPT = (
    'Extract all CEISA customs declaration fields from this shipping document. '
    'Return valid JSON with these keys: nomorBl, tglBl (YYYY-MM-DD), '
    'pelabuhan_muat, pelabuhan_bongkar, container_no, beratKotor, '
    'hs_code, namaKapal, voyageNumber.'
)

def load_image(path_str):
    path_str = str(path_str)
    if path_str.endswith('.pdf'):
        pages = convert_from_path(path_str, dpi=150, first_page=1, last_page=1)
        return pages[0].convert('RGB')
    return Image.open(path_str).convert('RGB')

def create_training_example(img_path, target_json_str):
    return {
        'messages': [
            {'role': 'user', 'content': [
                {'type': 'image'},
                {'type': 'text', 'text': EXTRACTION_PROMPT}
            ]},
            {'role': 'assistant', 'content': [
                {'type': 'text', 'text': target_json_str}
            ]}
        ],
        'image_path': str(img_path)
    }

# Load Ground Truth
gt_data = json.loads(GT_PATH.read_text()) if GT_PATH.exists() else {}
print(f'Ground Truth loaded: {len(gt_data)} documents')

dataset_examples = []

# A. Real augmented docs (TRAIN split only)
real_count = 0
if MANIFEST_PATH.exists():
    manifest = json.loads(MANIFEST_PATH.read_text())
    for item in manifest.get('train', []):
        doc_id = item.get('doc_id', '')
        if doc_id in gt_data:
            raw_path = str(item.get('path', ''))
            if raw_path.startswith('./dataset'):
                img_path = raw_path.replace('./dataset', str(NB0_INPUT / 'dataset'), 1)
            elif raw_path.startswith('/kaggle'):
                img_path = raw_path
            else:
                img_path = str(NB0_INPUT / 'dataset' / raw_path.lstrip('/'))
            if Path(img_path).exists():
                fields = gt_data[doc_id].get('ceisa_fields', gt_data[doc_id])
                dataset_examples.append(create_training_example(img_path, json.dumps(fields)))
                real_count += 1
            else:
                print(f'  SKIP (not found): {img_path}')
    print(f'Real augmented images: {real_count}')
else:
    print(f'MANIFEST NOT FOUND: {MANIFEST_PATH}. Only synthetic data will be used.')

# B. Synthetic docs
synth_count = 0
if SYNTHETIC_DIR.exists():
    for json_file in sorted(SYNTHETIC_DIR.glob('*.json'))[:300]:
        pdf_file = json_file.with_suffix('.pdf')
        if pdf_file.exists():
            gt_json = json.loads(json_file.read_text())
            dataset_examples.append(create_training_example(str(pdf_file), json.dumps(gt_json)))
            synth_count += 1
    print(f'Synthetic documents: {synth_count}')
else:
    print(f'SYNTHETIC_DIR NOT FOUND: {SYNTHETIC_DIR}')

print(f'\nTotal training examples: {len(dataset_examples)}')
train_dataset = Dataset.from_list(dataset_examples)


## 3. Custom Data Collator (Qwen2.5-VL Multimodal)

In [ ]:
class Qwen2VLDataCollator:
    """Collator for interleaved image-text inputs.
    
    IMPORTANT: No truncation! Qwen2.5-VL's processor validates that
    image token counts match between text and input_ids.
    Truncation would break this invariant.
    """
    def __init__(self, processor):
        self.processor = processor

    def __call__(self, examples):
        texts, images = [], []
        for ex in examples:
            text = self.processor.apply_chat_template(
                ex['messages'], tokenize=False, add_generation_prompt=False
            )
            texts.append(text)
            images.append(load_image(ex['image_path']))

        # NO truncation — critical for VLM image token alignment
        batch = self.processor(
            text=texts,
            images=images,
            return_tensors='pt',
            padding=True,
        )

        # Set labels: mask padding tokens with -100
        labels = batch['input_ids'].clone()
        if self.processor.tokenizer.pad_token_id is not None:
            labels[labels == self.processor.tokenizer.pad_token_id] = -100
        batch['labels'] = labels
        return batch

data_collator = Qwen2VLDataCollator(processor)
print('DataCollator ready.')


## 4. Training

In [ ]:
training_args = TrainingArguments(
    output_dir=str(OUT_DIR / 'checkpoints'),
    learning_rate=1e-4,
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    warmup_steps=20,
    weight_decay=0.01,
    logging_steps=5,
    save_strategy='no',
    remove_unused_columns=False,
    fp16=True,
    optim='adamw_torch',
    dataloader_num_workers=0,
    gradient_checkpointing=True,
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=data_collator,
)

print(f'Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}')
print(f'Total steps: ~{len(train_dataset) // (training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps)}')
print()
print('=== Starting Training ===')
train_result = trainer.train()
print('=== Training Complete ===')
print(f'Training loss: {train_result.training_loss:.4f}')

# Save LoRA adapter
save_path = OUT_DIR / 'best'
trainer.model.save_pretrained(save_path)
processor.save_pretrained(save_path)
print(f'Model saved to {save_path}')


## 5. Upload ke HuggingFace Hub

In [ ]:
HF_REPO_NAME = 'muhammadghiffari/olm-ocr-cipl-v1'

if hf_token:
    from huggingface_hub import HfApi
    api = HfApi(token=hf_token)
    api.create_repo(repo_id=HF_REPO_NAME, exist_ok=True)
    trainer.model.push_to_hub(HF_REPO_NAME, token=hf_token)
    processor.push_to_hub(HF_REPO_NAME, token=hf_token)
    print(f'\n✅ Model uploaded: https://huggingface.co/{HF_REPO_NAME}')
else:
    print('HF_TOKEN not found. Save adapter manually from:', OUT_DIR / 'best')
